In [24]:
#Load IOS feature selected dataset for modeling:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder



iOS_selected_features = pd.read_csv("../Processed_Data/Selected_FeaturesDatasets/iOS_SelectedFeatures.csv")

In [4]:
"stress" in iOS_selected_features.columns

True

In [5]:
#Preparing dataset for analysis:
#Select relevant columns for analysis:
y = iOS_selected_features['stress']
X = iOS_selected_features.drop(columns=['stress', 'uid', 'day'])  # Drop target, ID columns, and date
print("Final feature set columns:", X.columns)
print("Final feature set shape:", X.shape)  

Final feature set columns: Index(['Unnamed: 0.1', 'act_in_vehicle_ep_2', 'loc_study_audio_voice',
       'race_american indian/alaska native', 'loc_self_dorm_unlock_duration',
       'loc_self_dorm_dur', 'act_in_vehicle_ep_0', 'phq4-1', 'Unnamed: 0',
       'loc_self_dorm_audio_amp', 'sleep_duration', 'audio_amp_mean_ep_2',
       'unlock_num_ep_2', 'audio_amp_mean_ep_1', 'sleep_end',
       'loc_other_dorm_convo_duration', 'pam', 'loc_study_dur', 'phq4_score',
       'audio_amp_std_ep_2', 'phq4-2', 'race_other/hispanic',
       'audio_amp_std_ep_1', 'sse3-4', 'loc_self_dorm_still',
       'unlock_num_ep_0', 'act_still_ep_2', 'audio_voice_ep_3', 'race_black',
       'sse3-1', 'quality_loc', 'quality_audio', 'race_alaskan native/white',
       'social_level', 'audio_amp_mean_ep_3', 'sse3-3',
       'race_american indian/white', 'loc_study_unlock_duration', 'gender',
       'race_white', 'loc_max_dis_from_campus_ep_0', 'sse3_resp_mean',
       'race_more than one', 'avg_ema_spent_time', 

In [6]:
#Splitting data into test and train sets to prevent data leakage:

#Column that identifies groups (participants):
group_col = 'uid'

#80/20 training testing split, with one testing group:
gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=iOS_selected_features[group_col]))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = iOS_selected_features[group_col].iloc[train_idx]

#Checking stress label distribution to ensure the groups are stratified:
print("Train distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest distribution:")
print(y_test.value_counts(normalize=True))

Train distribution:
stress
2.0    0.335620
3.0    0.292450
1.0    0.173687
4.0    0.144979
5.0    0.053264
Name: proportion, dtype: float64

Test distribution:
stress
2.0    0.359751
3.0    0.285915
1.0    0.171950
4.0    0.142055
5.0    0.040329
Name: proportion, dtype: float64


In [7]:
# Stratified Group K-Fold
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

**Random Forest Classifier**

In [8]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt', 
    class_weight=None,   # key addition for classification
    random_state=42,
    n_jobs=-1
)

RF_fold_f1 = []
RF_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx]
    y_fold_val = y_train.iloc[val_idx]
    
    #Impute missing values (important to do before SMOTE to avoid errors):
    imputer = SimpleImputer(strategy='median')
    X_fold_train_imputed = imputer.fit_transform(X_fold_train)
    X_fold_val_imputed = imputer.transform(X_fold_val)
    
    # Apply SMOTE to imputed training data
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train_imputed, y_fold_train)
    
    # Train
    rf_model.fit(X_resampled, y_resampled)
    
    # Predict (on imputed validation set)
    val_preds = rf_model.predict(X_fold_val_imputed)
    
    # Classification metrics (weighted F1 and accuracy)
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    RF_fold_f1.append(f1)
    RF_fold_acc.append(acc)
    
    print(f"Fold {fold+1} - F1: {f1:.4f}, Accuracy: {acc:.4f}")
    
    #
    print("Classification Report:")
    print(classification_report(y_fold_val, val_preds))


print(f"\nAverage F1 across folds: {sum(RF_fold_f1)/len(RF_fold_f1):.4f}")
print(f"Average Accuracy across folds: {sum(RF_fold_acc)/len(RF_fold_acc):.4f}")

Fold 1 - F1: 0.4373, Accuracy: 0.4356
Classification Report:
              precision    recall  f1-score   support

         1.0       0.56      0.54      0.55       793
         2.0       0.46      0.40      0.43      1537
         3.0       0.44      0.43      0.44      1326
         4.0       0.32      0.35      0.33       679
         5.0       0.30      0.55      0.39       247

    accuracy                           0.44      4582
   macro avg       0.42      0.45      0.43      4582
weighted avg       0.44      0.44      0.44      4582

Fold 2 - F1: 0.4693, Accuracy: 0.4743
Classification Report:
              precision    recall  f1-score   support

         1.0       0.53      0.49      0.51       789
         2.0       0.46      0.54      0.50      1547
         3.0       0.45      0.45      0.45      1348
         4.0       0.36      0.24      0.29       610
         5.0       0.68      0.72      0.70       252

    accuracy                           0.47      4546
   macro 

**XGBoost Classifier**

In [23]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softmax',  # multi-class classification
    num_class=5,                
    random_state=42,
    n_jobs=-1
)

XGB_fold_f1 = []
XGB_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    # Split
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    y_fold_val = y_train.iloc[val_idx]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_fold_train = imputer.fit_transform(X_fold_train)
    X_fold_val = imputer.transform(X_fold_val)
   
    # Adjust labels to start from 0 for XGBoost
    y_fold_train = y_fold_train - 1  
    y_fold_val = y_fold_val - 1  
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)
    

    # Train
    xgb_model.fit(X_resampled, y_resampled)
    
    # Predict
    val_preds = xgb_model.predict(X_fold_val)
    
    # Metrics
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    XGB_fold_f1.append(f1)
    XGB_fold_acc.append(acc)
    
    print(f"\nFold {fold+1} - F1: {f1:.4f}, Accuracy: {acc:.4f}")
    print(classification_report(y_fold_val, val_preds))

print(f"\nAverage F1 across folds: {np.mean(XGB_fold_f1):.4f}")
print(f"Average Accuracy across folds: {np.mean(XGB_fold_acc):.4f}")


Fold 1 - F1: 0.4525, Accuracy: 0.4537
              precision    recall  f1-score   support

         0.0       0.56      0.51      0.53       793
         1.0       0.47      0.46      0.46      1537
         2.0       0.42      0.50      0.46      1326
         3.0       0.40      0.31      0.35       679
         4.0       0.37      0.40      0.38       247

    accuracy                           0.45      4582
   macro avg       0.44      0.43      0.44      4582
weighted avg       0.46      0.45      0.45      4582


Fold 2 - F1: 0.4726, Accuracy: 0.4771
              precision    recall  f1-score   support

         0.0       0.54      0.39      0.46       789
         1.0       0.46      0.58      0.51      1547
         2.0       0.45      0.47      0.46      1348
         3.0       0.38      0.25      0.30       610
         4.0       0.81      0.68      0.74       252

    accuracy                           0.48      4546
   macro avg       0.53      0.48      0.49      4546

**Multinomial Logistic Regression**

In [10]:
lr_model = LogisticRegression(
    solver='lbfgs',
    max_iter=1000,
    class_weight=None,  
    random_state=42
)

LR_fold_f1 = []
LR_fold_acc = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train, groups=groups_train)):
    
    # Split + numeric only
    X_fold_train = X_train.iloc[train_idx].select_dtypes(include=[np.number])
    y_fold_train = y_train.iloc[train_idx]
    X_fold_val = X_train.iloc[val_idx].select_dtypes(include=[np.number])
    y_fold_val = y_train.iloc[val_idx]
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_fold_train = imputer.fit_transform(X_fold_train)
    X_fold_val = imputer.transform(X_fold_val)
    
    # Scale ( to avoid domination by features with larger ranges)
    scaler = StandardScaler()
    X_fold_train = scaler.fit_transform(X_fold_train)
    X_fold_val = scaler.transform(X_fold_val)
    
    # SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X_fold_train, y_fold_train)
    
    # Train
    lr_model.fit(X_resampled, y_resampled)
    
    # Predict
    val_preds = lr_model.predict(X_fold_val)
    
    # Metrics
    f1 = f1_score(y_fold_val, val_preds, average='weighted')
    acc = accuracy_score(y_fold_val, val_preds)
    
    LR_fold_f1.append(f1)
    LR_fold_acc.append(acc)
    
    print(f"\nFold {fold+1} - F1: {f1:.4f}, Accuracy: {acc:.4f}")
    print(classification_report(y_fold_val, val_preds))

print(f"\nAverage F1 across folds: {np.mean(LR_fold_f1):.4f}")
print(f"Average Accuracy across folds: {np.mean(LR_fold_acc):.4f}")


Fold 1 - F1: 0.3328, Accuracy: 0.3431
              precision    recall  f1-score   support

         1.0       0.46      0.69      0.55       793
         2.0       0.44      0.26      0.33      1537
         3.0       0.40      0.19      0.26      1326
         4.0       0.21      0.30      0.25       679
         5.0       0.19      0.70      0.30       247

    accuracy                           0.34      4582
   macro avg       0.34      0.43      0.34      4582
weighted avg       0.39      0.34      0.33      4582


Fold 2 - F1: 0.3918, Accuracy: 0.3968
              precision    recall  f1-score   support

         1.0       0.44      0.58      0.50       789
         2.0       0.44      0.35      0.39      1547
         3.0       0.43      0.31      0.36      1348
         4.0       0.26      0.32      0.29       610
         5.0       0.34      0.73      0.46       252

    accuracy                           0.40      4546
   macro avg       0.38      0.46      0.40      4546

**Personlized Models**

In [12]:
# checking distribution of stress labels across participants to understand class balance:
iOS_selected_features.groupby('uid')['stress'].value_counts(normalize=True)

counts = iOS_selected_features['uid'].value_counts()

print(counts.describe())
print("\nSmallest participants:")
print(counts.sort_values().head(10))

enough_data = counts[counts >= 30]
small_data = counts[counts < 30]
#Done to make sure that we have enough samples per participant for a 

print(f"Participants with >=30 samples: {len(enough_data)}")
print(f"Participants with <30 samples: {len(small_data)}")

count    196.000000
mean     142.193878
std       69.395878
min        1.000000
25%       88.750000
50%      159.000000
75%      191.000000
max      350.000000
Name: count, dtype: float64

Smallest participants:
uid
115     1
10      1
73      9
119    10
18     13
41     13
154    18
151    19
215    20
46     25
Name: count, dtype: int64
Participants with >=30 samples: 183
Participants with <30 samples: 13


In [ ]:
#Random Forest personalized models for each participant 

unique_participants = iOS_selected_features['uid'].unique()

personalized_results = {}

for participant in unique_participants:
    
    participant_data = iOs_selected_features[
        android_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip if too little data
    if len(participant_data) < 10:
        continue
    
    # Skip if only one class
    if y_participant.nunique() < 2:
        continue
    
    # Train-test split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Keep numeric features only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute missing values
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    # Random Forest model
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        class_weight='balanced',  # helps imbalance per participant
        random_state=42,
        n_jobs=-1
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])

print(f"\nAverage F1 across personalized RF models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized RF models: {avg_acc:.4f}")

f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base


Average F1 across personalized RF models: 0.4654
Average Accuracy across personalized RF models: 0.4913
Min F1: 0.0
Max F1: 1.0


/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


In [ ]:

# Creating personalized XGBoost models for each participant
unique_participants = iOS_selected_features['uid'].unique()
personalized_results = {}
skipped_participants = []
processed_participants = []



for participant in unique_participants:
    
    participant_data = iOS_selected_features[iOS_selected_features['uid'] == participant]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_participant)

    if len(participant_data) < 10:
        skipped_participants.append((participant, "too few samples"))
        continue
    

    # Skip bad class distributions
    if min(np.bincount(y_encoded)) < 2:
        skipped_participants.append((participant, "class with <2 samples"))
        continue
    
    processed_participants.append(participant)
    

    # Train-test split
    X_train_p, X_test_p, y_train_adj, y_test_adj = train_test_split(
    X_participant,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded  #need to stratify to ensure all classes are represented in train/test splits
    )
    
    # Keep numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
  
    # Model
    model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='multi:softmax',
        # num_class = len(np.unique(y_encoded)),
        random_state=42,
        n_jobs=-1
    )
    
    #predict
    model.fit(X_train_p, y_train_adj)
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_adj, preds, average='weighted')
    acc = accuracy_score(y_test_adj, preds)
    
    personalized_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Average performance
avg_f1 = np.mean([res['f1'] for res in personalized_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_results.values()])
print(f"Total participants: {len(unique_participants)}")
print(f"Processed participants: {len(processed_participants)}")
print(f"Skipped participants: {len(skipped_participants)}")

print("\nSkip reasons:")
for p, reason in skipped_participants[:10]:  # show first 10
    print(p, "-", reason)

print(f"\nAverage F1 across personalized models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized models: {avg_acc:.4f}")
f1_scores = [res['f1'] for res in personalized_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base

Total participants: 196
Processed participants: 149
Skipped participants: 47

Skip reasons:
9 - class with <2 samples
10 - too few samples
11 - class with <2 samples
12 - class with <2 samples
13 - class with <2 samples
14 - class with <2 samples
15 - class with <2 samples
18 - class with <2 samples
22 - class with <2 samples
25 - class with <2 samples

Average F1 across personalized models: 0.4310
Average Accuracy across personalized models: 0.4605
Min F1: 0.1111111111111111
Max F1: 0.712568058076225


In [19]:
#Multinomial Logistic Regression personalized models for each participant

unique_participants = iOS_selected_features['uid'].unique()

personalized_lr_results = {}

for participant in unique_participants:
    
    participant_data = iOS_selected_features[
        iOS_selected_features['uid'] == participant
    ]
    
    X_participant = participant_data.drop(columns=['stress', 'uid', 'day'])
    y_participant = participant_data['stress']
    
    # Skip checks
    if len(participant_data) < 10:
        continue
    
    if y_participant.nunique() < 2:
        continue
    
    # Split
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
        X_participant, y_participant, test_size=0.2, random_state=42
    )
    
    # Numeric only
    X_train_p = X_train_p.select_dtypes(include=[np.number])
    X_test_p = X_test_p.select_dtypes(include=[np.number])
    
    # Impute
    imputer = SimpleImputer(strategy='median')
    X_train_p = imputer.fit_transform(X_train_p)
    X_test_p = imputer.transform(X_test_p)
    
    #  Scale 
    scaler = StandardScaler()
    X_train_p = scaler.fit_transform(X_train_p)
    X_test_p = scaler.transform(X_test_p)
    
    # Model
    model = LogisticRegression(
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',  # helps within-user imbalance
        random_state=42
    )
    
    # Train
    model.fit(X_train_p, y_train_p)
    
    # Predict
    preds = model.predict(X_test_p)
    
    # Metrics
    f1 = f1_score(y_test_p, preds, average='weighted')
    acc = accuracy_score(y_test_p, preds)
    
    personalized_lr_results[participant] = {
        'f1': f1,
        'accuracy': acc
    }

# Averages
avg_f1 = np.mean([res['f1'] for res in personalized_lr_results.values()])
avg_acc = np.mean([res['accuracy'] for res in personalized_lr_results.values()])

print(f"\nAverage F1 across personalized LR models: {avg_f1:.4f}")
print(f"Average Accuracy across personalized LR models: {avg_acc:.4f}")

#  variability
f1_scores = [res['f1'] for res in personalized_lr_results.values()]
print("Min F1:", np.min(f1_scores))
print("Max F1:", np.max(f1_scores))

/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base


Average F1 across personalized LR models: 0.4404
Average Accuracy across personalized LR models: 0.4347
Min F1: 0.0
Max F1: 1.0


/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:641: UserWarning: Skipping features without any observed values: ['loc_self_dorm_audio_amp' 'audio_amp_mean_ep_2' 'audio_amp_mean_ep_1'
 'audio_amp_std_ep_2' 'audio_amp_std_ep_1' 'audio_amp_mean_ep_3'
 'audio_amp_std_ep_0']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/sethunair/Desktop/Winter 2026/619/BMEN619_StudentLife/.venv/lib/python3.12/site-packages/sklearn/impute/_base